# 🧠 Reasoning Fine-tuning Guide
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/togethercomputer/together-cookbook/blob/main/Finetuning/Reasoning_Finetuning.ipynb)

Reasoning fine-tuning teaches models to think step-by-step before producing a final answer. By providing a `reasoning` field alongside assistant responses, you shape how a model works through problems internally — similar to the `<think>` blocks seen in reasoning models like DeepSeek-R1 and Qwen3.

This notebook provides a step-by-step guide to fine-tuning a reasoning model using the Together AI platform. We'll use a subset of the [OpenR1-Math-220k](https://huggingface.co/datasets/open-r1/OpenR1-Math-220k) dataset, which contains math problems paired with verified chain-of-thought reasoning traces distilled from DeepSeek-R1.

We will cover:

1. **Dataset Exploration:** Loading and understanding the reasoning dataset.
2. **Data Transformation:** Converting the dataset to Together AI's reasoning fine-tuning format.
3. **Data Upload:** Validating and uploading the prepared dataset to Together AI.
4. **Fine-tuning Job Launch:** Configuring and starting a LoRA fine-tuning job.
5. **Job Monitoring:** Checking the status and progress of your fine-tuning job.
6. **Inference:** Comparing the base model vs. fine-tuned model on math reasoning tasks.

By following this guide, you'll learn how to create models with improved chain-of-thought reasoning using Together AI.

## Setup and Installation
---
First, install the necessary Python libraries:
- `together`: The official Together AI Python client for interacting with the API.
- `datasets`: A library from Hugging Face for easily downloading and manipulating datasets.
- `tqdm`: For progress bars.

In [ ]:
!pip install -qU together datasets tqdm

In [ ]:
import os
import re
import json
from together import Together
from datasets import load_dataset
from tqdm.auto import tqdm

# Setup Together AI client
TOGETHER_API_KEY = os.getenv("TOGETHER_API_KEY")
WANDB_API_KEY = os.getenv("WANDB_API_KEY")  # Optional: for logging to WandB

client = Together(api_key=TOGETHER_API_KEY)

## 1. Dataset Exploration
---
We'll use the [OpenR1-Math-220k](https://huggingface.co/datasets/open-r1/OpenR1-Math-220k) dataset from Hugging Face's Open R1 project. This dataset contains:
- **93,700 math problems** (default subset) with multiple reasoning traces
- Each problem has **verified chain-of-thought solutions** distilled from DeepSeek-R1
- Solutions include `<think>` blocks with step-by-step reasoning followed by a final answer
- Correctness is verified via mathematical answer checking (`correctness_math_verify`)

In [ ]:
# Load the default subset of OpenR1-Math-220k
dataset = load_dataset("open-r1/OpenR1-Math-220k", "default", split="train")
print(f"Dataset size: {len(dataset)}")
print(f"Columns: {dataset.column_names}")

In [ ]:
# Examine a sample
sample = dataset[0]
print(f"Problem:\n{sample['problem'][:300]}...")
print(f"\nAnswer: {sample['answer']}")
print(f"\nNumber of generated solutions: {len(sample['generations'])}")
print(f"Correctness flags: {sample['correctness_math_verify']}")

In [ ]:
# Look at a solution — it contains <think> blocks with reasoning
# Pick the first correct solution
correct_idx = next(
    i for i, flag in enumerate(sample["correctness_math_verify"]) if flag
)
solution = sample["generations"][correct_idx]
print(f"Solution (first 500 chars):\n{solution[:500]}...")
print(f"\n...\n\nSolution (last 300 chars):\n{solution[-300:]}")

Each solution follows the pattern:
- `<think>...</think>` — the model's internal chain-of-thought reasoning
- Text after `</think>` — the final answer presented to the user

For Together AI's reasoning fine-tuning format, we need to split these into separate `reasoning` and `content` fields.

## 2. Data Transformation to Reasoning Format
---
Together AI's reasoning fine-tuning requires assistant messages to include a `reasoning` field for the chain-of-thought and a `content` field for the final answer.

### Required Format
```json
{
  "messages": [
    {"role": "user", "content": "What is 25 * 4 + 10?"},
    {
      "role": "assistant",
      "reasoning": "Let me work through this step by step. First, 25 * 4 = 100. Then 100 + 10 = 110.",
      "content": "The answer is 110."
    }
  ]
}
```

### Key Requirements:
- Assistant messages include a `reasoning` field (the chain-of-thought) and a `content` field (the final answer)
- The `reasoning` field captures the model's internal thinking process
- The `content` field contains only the response shown to the user

We'll **subsample 200 training and 50 validation examples** since this is a demo. For production use cases, you'd use significantly more data.

In [ ]:
SYSTEM_PROMPT = "You are a math reasoning assistant. Think through problems step by step before giving your final answer."


def parse_think_tags(solution: str) -> tuple[str, str] | None:
    """
    Parse a solution string containing <think>...</think> tags.

    Args:
        solution: The full solution text with think tags.

    Returns:
        Tuple of (reasoning, final_answer), or None if parsing fails.
    """
    match = re.search(r"<think>(.*?)</think>(.*)", solution, re.DOTALL)
    if not match:
        return None

    reasoning = match.group(1).strip()
    final_answer = match.group(2).strip()

    if not reasoning or not final_answer:
        return None

    return reasoning, final_answer


def convert_to_reasoning_format(example: dict) -> dict | None:
    """
    Convert an OpenR1-Math-220k example to Together AI's reasoning fine-tuning format.

    Selects the first verified-correct solution, parses the <think> tags, and
    structures it into the reasoning format.

    Args:
        example: A single example from the dataset.

    Returns:
        Dictionary in reasoning fine-tuning format, or None if conversion fails.
    """
    # Find the first correct solution
    correct_idx = None
    for i, flag in enumerate(example["correctness_math_verify"]):
        if flag:
            correct_idx = i
            break

    if correct_idx is None:
        return None

    solution = example["generations"][correct_idx]
    parsed = parse_think_tags(solution)
    if parsed is None:
        return None

    reasoning, final_answer = parsed

    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["problem"]},
            {
                "role": "assistant",
                "reasoning": reasoning,
                "content": final_answer,
            },
        ]
    }

In [ ]:
# Test the conversion on our sample
converted = convert_to_reasoning_format(sample)

print("Converted format:")
print(json.dumps(converted, indent=2)[:2000] + "...")

In [ ]:
# Convert and subsample the dataset
# We use a small subset for this demo — scale up for production use cases
TRAIN_SIZE = 200
VAL_SIZE = 50

# Shuffle the dataset so we get a diverse sample
shuffled = dataset.shuffle(seed=42)

print("Converting training examples...")
train_data = []
val_data = []
for example in tqdm(shuffled):
    converted = convert_to_reasoning_format(example)
    if converted is None:
        continue

    if len(train_data) < TRAIN_SIZE:
        train_data.append(converted)
    elif len(val_data) < VAL_SIZE:
        val_data.append(converted)
    else:
        break

print(f"Converted {len(train_data)} training examples")
print(f"Converted {len(val_data)} validation examples")

In [ ]:
# Save to JSONL files
def save_jsonl(data: list, filepath: str):
    """Save a list of dictionaries to a JSONL file."""
    with open(filepath, "w") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")
    print(f"Saved {len(data)} examples to {filepath}")

save_jsonl(train_data, "reasoning_train.jsonl")
save_jsonl(val_data, "reasoning_val.jsonl")

## 3. Upload Data to Together AI
---
Now we'll validate and upload our prepared datasets to Together AI. The `check=True` parameter validates the file format before uploading.

In [ ]:
# Check the training file format
from together.utils import check_file

print("Validating training file...")
train_report = check_file("reasoning_train.jsonl")
print(json.dumps(train_report, indent=2))

assert train_report["is_check_passed"] == True, "Training file validation failed!"

In [ ]:
# Upload training file
print("Uploading training file...")
train_file = client.files.upload("reasoning_train.jsonl", check=True)
print(f"Training file ID: {train_file.id}")

In [ ]:
# Upload validation file
print("Uploading validation file...")
val_file = client.files.upload("reasoning_val.jsonl", check=True)
print(f"Validation file ID: {val_file.id}")

## 4. Launch Fine-tuning Job
---
Now we'll launch a LoRA fine-tuning job for reasoning.

### Key Parameters

| Parameter | Description | Our Value |
|-----------|-------------|-----------|
| `model` | Base model to fine-tune | `Qwen/Qwen3-8B` |
| `lora` | Use LoRA (recommended) | `True` |
| `n_epochs` | Training epochs | `3` |
| `learning_rate` | Weight update rate | `1e-5` |
| `n_checkpoints` | Checkpoints to save | `1` |
| `suffix` | Custom model name suffix | `reasoning-demo` |

> **Note:** These hyperparameters are tuned for this small demo dataset. For production, you'd use more data and may want to adjust epochs and learning rate accordingly.

🔗 See the [Reasoning Fine-tuning documentation](https://docs.together.ai/docs/reasoning-fine-tuning) for the full list of supported models and parameters.

In [ ]:
# Launch the LoRA fine-tuning job
ft_response = client.fine_tuning.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model="Qwen/Qwen3-8B",
    lora=True,
    n_epochs=3,
    learning_rate=1e-5,
    n_checkpoints=1,
    suffix="reasoning-demo",
    wandb_api_key=WANDB_API_KEY,
)

print(f"Fine-tuning job created!")
print(f"Job ID: {ft_response.id}")

## 5. Monitor Fine-tuning Job
---
You can monitor your fine-tuning job's progress using the Together AI API or the [dashboard](https://api.together.ai/jobs).

Your job will progress through several states: Pending → Queued → Running → Uploading → Completed.

Available methods:
- `client.fine_tuning.retrieve(id)` — Get job status
- `client.fine_tuning.list_events(id=job_id)` — Get job logs
- `client.fine_tuning.cancel(id=job_id)` — Cancel a job
- `client.fine_tuning.list()` — List all jobs

In [ ]:
# Check the status of the fine-tuning job
status = client.fine_tuning.retrieve(ft_response.id)
print(f"Job Status: {status.status}")

In [ ]:
# View training logs/events
events = client.fine_tuning.list_events(id=ft_response.id)
for event in events.data:
    print(event.message)

In [ ]:
# Wait for job completion (optional — you can also check the dashboard)
import time

while True:
    status = client.fine_tuning.retrieve(ft_response.id)
    print(f"Status: {status.status}")

    if status.status in ["completed", "failed", "cancelled"]:
        break

    time.sleep(60)  # Check every minute

print(f"\nFinal status: {status.status}")
if status.status == "completed":
    print(f"Fine-tuned model: {status.output_name}")

🔗 You can also monitor your job on the [Together AI Fine-tuning Dashboard](https://api.together.ai/jobs) and view WandB logs if you provided an API key.

## 6. Inference — Base Model vs. Fine-tuned Model
---
Once training is complete, let's compare the base model against our fine-tuned model on math reasoning tasks. We'll look at both the chain-of-thought reasoning and the final answer.

In [ ]:
# Get the fine-tuned model name
finetuned_model = status.output_name
print(f"Fine-tuned model: {finetuned_model}")

In [ ]:
# Define test math problems (not in our training set)
test_problems = [
    "A store sells notebooks for $4 each and pens for $1.50 each. Sarah buys 3 notebooks and 5 pens. If she pays with a $50 bill, how much change does she receive?",
    "A train travels at 60 mph for the first 2 hours, then at 80 mph for the next 3 hours. What is the average speed for the entire journey?",
    "In a class of 40 students, 25 play basketball and 20 play soccer. If 10 students play both sports, how many students play neither sport?",
]

In [ ]:
def test_reasoning(model_name: str, problem: str) -> dict:
    """
    Test a model's reasoning capability on a math problem.

    Args:
        model_name: Together AI model identifier.
        problem: Math problem to solve.

    Returns:
        Dictionary with reasoning (if available) and final answer.
    """
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": problem},
        ],
        max_tokens=4096,
    )

    choice = response.choices[0].message
    result = {"content": choice.content}

    # Check for reasoning content (returned by reasoning-tuned models)
    if hasattr(choice, "reasoning_content") and choice.reasoning_content:
        result["reasoning"] = choice.reasoning_content

    return result

Below we compare the fine-tuned model against the base model. The fine-tuned model should produce structured chain-of-thought reasoning in a separate `reasoning_content` field, while the base model will include all reasoning inline in its response.

In [ ]:
# Compare base model vs fine-tuned model
models_to_test = {
    "Base model (Qwen3-8B)": "Qwen/Qwen3-8B",
    "Fine-tuned model": finetuned_model,
}

for problem in test_problems:
    print(f"Problem: {problem}")
    print("=" * 60)

    for label, model_id in models_to_test.items():
        print(f"\n{label}:")
        print("-" * 40)
        result = test_reasoning(model_id, problem)

        if "reasoning" in result:
            reasoning_preview = result["reasoning"][:300]
            print(f"  Reasoning: {reasoning_preview}...")
        else:
            print("  Reasoning: (not separated — included in response)")

        content = result["content"] or "(no response)"
        print(f"  Answer: {content[:500]}")

    print("\n")

## Summary
---
In this notebook, we demonstrated how to:

1. **Load and explore** a reasoning dataset with chain-of-thought traces from Open R1
2. **Transform data** into Together AI's reasoning format by parsing `<think>` tags into the `reasoning` field
3. **Upload and validate** the dataset using the Together AI Python client
4. **Launch a LoRA fine-tuning job** on Qwen3-8B for reasoning
5. **Monitor training progress** via the API and dashboard
6. **Compare inference** between the base model and fine-tuned model

### Key Takeaways
- Reasoning fine-tuning uses the `reasoning` field in assistant messages to capture the model's chain-of-thought thinking process
- The `content` field contains only the final answer shown to the user, keeping the reasoning separate
- Training on verified reasoning traces (from models like DeepSeek-R1) teaches models to produce structured, step-by-step thinking
- LoRA fine-tuning is recommended — it's faster, cheaper, and enables serverless inference

### Next Steps
- Scale up to more training data for improved reasoning quality
- Try DPO preference tuning to teach the model to prefer better reasoning chains over worse ones
- Experiment with different base models (Qwen3-4B, Qwen3-32B, etc.)
- Apply reasoning fine-tuning to other domains like code, science, or logic puzzles
- Deploy your fine-tuned model on a [dedicated endpoint](https://docs.together.ai/docs/dedicated-endpoints) for production use

🔗 For more details, see the [Reasoning Fine-tuning documentation](https://docs.together.ai/docs/reasoning-fine-tuning).